In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import torch

from ultralytics.engine.results import Results
from ultralytics.utils import YAML, nms, ops
from ultralytics.utils.checks import check_yaml


class YOLO26SegONNX:
    def __init__(self, onnx_model: str, conf: float = 0.25, iou: float = 0.7, imgsz=640):
        available = ort.get_available_providers()
        providers = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in available]
        self.session = ort.InferenceSession(onnx_model, providers=providers or available)

        self.imgsz = (imgsz, imgsz) if isinstance(imgsz, int) else imgsz
        self.conf = conf
        self.iou = iou

        # COCO pretrained 기준
        self.classes = YAML.load(check_yaml("coco8.yaml"))["names"]

        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]

        print("Input:", self.input_name, self.session.get_inputs()[0].shape)
        print("Outputs:", [(o.name, o.shape) for o in self.session.get_outputs()])

    def __call__(self, img: np.ndarray):
        prep = self.preprocess(img, self.imgsz)
        outputs = self.session.run(self.output_names, {self.input_name: prep})

        # print("outputs len:", len(outputs))
        # for i, o in enumerate(outputs):
        #     print(f"output[{i}] shape:", o.shape)

        return self.postprocess(img, prep, outputs)

    def letterbox(self, img: np.ndarray, new_shape=(640, 640)):
        shape = img.shape[:2]  # (h, w)
        r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

        new_unpad = (round(shape[1] * r), round(shape[0] * r))
        dw = (new_shape[1] - new_unpad[0]) / 2
        dh = (new_shape[0] - new_unpad[1]) / 2

        if shape[::-1] != new_unpad:
            img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)

        top, bottom = round(dh - 0.1), round(dh + 0.1)
        left, right = round(dw - 0.1), round(dw + 0.1)

        img = cv2.copyMakeBorder(
            img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114)
        )
        return img

    def preprocess(self, img: np.ndarray, new_shape):
        img = self.letterbox(img, new_shape)
        img = img[..., ::-1].transpose(2, 0, 1)[None]  # BGR -> RGB, HWC -> BCHW
        img = np.ascontiguousarray(img).astype(np.float32) / 255.0
        return img

    def postprocess(self, img: np.ndarray, prep_img: np.ndarray, outs: list):
        """
        YOLO26-seg end-to-end ONNX 출력 형식:
        output0: [1, 300, 38] = [x1,y1,x2,y2,conf,class_id,32 mask coeff]
        output1: [1, 32, 160, 160] = mask protos
        """
        pred = torch.from_numpy(outs[0][0]).float()   # [300, 38]
        proto = torch.from_numpy(outs[1][0]).float()  # [32, 160, 160]

        # confidence filter만 적용 (YOLO26 e2e는 NMS-free)
        pred = pred[pred[:, 4] > self.conf]

        if len(pred) == 0:
            return [Results(orig_img=img, path="", names=self.classes, boxes=torch.empty((0, 6)))]

        # 분리
        boxes = pred[:, 0:4].clone()          # xyxy
        scores = pred[:, 4].clone()           # conf
        class_ids = pred[:, 5].clone()        # class_id
        mask_coeffs = pred[:, 6:38].clone()   # 32-dim

        # 좌표를 원본 이미지 크기로 복원
        boxes = ops.scale_boxes(prep_img.shape[2:], boxes, img.shape)

        # 마스크 생성
        masks = self.process_mask(proto, mask_coeffs, boxes, img.shape[:2])

        # Results 형식: [x1, y1, x2, y2, conf, cls]
        boxes_for_result = torch.cat(
            [boxes, scores.unsqueeze(1), class_ids.unsqueeze(1)],
            dim=1
        )

        result = Results(
            orig_img=img,
            path="",
            names=self.classes,
            boxes=boxes_for_result,
            masks=masks
        )
        return [result]

    def process_mask(self, protos: torch.Tensor, masks_in: torch.Tensor, bboxes: torch.Tensor, shape):
        """
        protos: [32, 160, 160]
        masks_in: [N, 32]
        bboxes: [N, 4] in original image scale
        """
        if protos.ndim == 4:
            protos = protos[0]

        if protos.ndim != 3:
            raise ValueError(f"Unexpected protos shape: {tuple(protos.shape)}")

        c, mh, mw = protos.shape
        if masks_in.shape[1] != c:
            raise ValueError(f"Mask dim mismatch: masks_in={tuple(masks_in.shape)}, protos={tuple(protos.shape)}")

        # coeff x proto
        masks = (masks_in @ protos.reshape(c, -1)).reshape(-1, mh, mw)

        # 원본 이미지 크기로 스케일
        masks = ops.scale_masks(masks[None], shape)[0]

        # 박스로 crop
        masks = ops.crop_mask(masks, bboxes)

        return masks.gt_(0.0)

if __name__ == "__main__":
    model = YOLO26SegONNX("yolo26n-seg.onnx", conf=0.25, iou=0.7, imgsz=640)

    cap = cv2.VideoCapture(0)

    if cap.isOpened():
        print("실시간 세그멘테이션 시작")
        print("종료하려면 q를 누르세요.")

        while True:
            ret, frame = cap.read()
            if not ret:
                print("프레임을 읽을 수 없습니다.")
                break

            try:
                results = model(frame)
                annotated = results[0].plot() if len(results) else frame
            except Exception as e:
                annotated = frame.copy()
                cv2.putText(
                    annotated,
                    f"Error: {str(e)}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2,
                    cv2.LINE_AA,
                )

            cv2.imshow("YOLO26 Segmentation Realtime (ONNXRuntime)", annotated)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()